# 週末のシグナル確認：SOMPO ホールディングス（8630）／ウィリアムズ%R(14) < -80

毎週末（金曜の取引終了後〜日曜）に上から順に実行し、**SOMPO にシグナルが出ているか**を確認するノートブックです。あわせて、**上昇確率を上げる・下げる条件が同時に出ているか**を確認し、その条件が過去に本当に効いていたかを検証できます。

**判定の型（これまでの検証と同じ）**
- 週の最終営業日の終値で **ウィリアムズ%R(14) < -80** なら「シグナルあり」
- 翌週の最初の営業日の寄りで買い、最長でその週の最終営業日の引けで売る

**分析時点（2008〜2026/10）の結果**
- シグナル172回、翌週の上昇確率64.0%
- 片道0.1%のコスト控除後で、1回あたり平均 +0.84%
- 信用50万円・複利で2011/10〜2026/10 に仕掛けた場合の試算：損切りなし +78万円、損切り7% +40万円（税引前）

**同時に見る条件**

| 向き | 条件 | 根拠 |
|---|---|---|
| ＋ | 陽線3日連続、25日線乖離率 ≤ -10%、パラボリックSAR買い転換、市場全体が週間 -3% 以下 | 全銘柄で前半・後半とも上昇確率が高かった |
| － | 終値 > 75日線、市場全体が75日線より上 | 全銘柄・他の保険株で上昇確率が低かった |
| ？ | 浅い下げ（週間 -3% 未満）、市場は上昇でこの銘柄だけ下落、業種ぐるみの下げ、この銘柄だけの下げ | SOMPO だけで目立った（偶然の可能性が高い） |

**参考点** ＝ ＋の条件の数 － －の条件の数（？は数えない）

> ⚠️ **注意**
> - SOMPO は、過去の成績が上位だったから選んだ銘柄です。多数の銘柄を比べたことを補正すると偶然と区別できず、実際より良く見えている可能性が高いです。
> - 条件も参考点も、売買ルールとして確かめたものではありません。
> - データは Yahoo Finance（非公式）です。対象は今上場している銘柄だけなので、成績が良く見えやすい偏り（生存者バイアス）があります。決算発表・権利落ちなどの予定は考慮していません。
> - このノートブックは判断材料をそろえるもので、売買の推奨ではありません。

## 1. 設定

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

TICKER = "8630.T"
NAME = "SOMPOホールディングス"
REFRESH = True        # 週末に最新データへ更新するなら True（全約490銘柄を取り直すため数分かかる）
CAPITAL = 500_000     # 信用の保証金（円）。レバレッジ1倍で100株単位の株数を計算する
INTEREST = 0.028      # 信用の買方金利（年率）。お使いの証券会社の料率に合わせる
ENTRY_PRICE = None    # 寄りで約定した後に損切りの目安を出し直すときは買値を入れる（例: 6900.0）
RECENT_YEARS = 3      # 過去シグナル一覧に表示する年数
CHECK_CONDITIONS = True  # 5. の条件の検証（全銘柄で集計するため1〜2分かかる）

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

from weekpick import data as WD
from weekpick import timing as T
from weekpick.universe import load_universe

for f in ["IPAGothic", "IPAexGothic", "Hiragino Sans", "Yu Gothic", "Meiryo", "Noto Sans CJK JP"]:
    if any(f == x.name for x in font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = f
        break
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
SIGNAL, CONDS = "wr14", T.WR_CONDITIONS
SG = T.SIGNALS[SIGNAL]

## 2. データ更新

市場全体の値動き・同業種の状況を計算するため、TOPIX500 相当の全銘柄を読み込みます。銘柄一覧（`data/universe_jpx.csv`）が古くなったら、`python -m weekpick universe` で更新してください。

In [ ]:
u = load_universe("topix500")
if REFRESH:
    failed = WD.fetch(list(u["ticker"]), refresh=True)
    print("取得失敗:", failed or "なし")
p = WD.load_panel(list(u["ticker"]))
for n in p.notes:
    print("・", n)
last = p.close.index[-1]
print(f"最新データ: {last:%Y-%m-%d}（{'月火水木金土日'[last.weekday()]}）")
if last.weekday() != 4:
    print("⚠️ 最新データが金曜ではありません。週の途中のデータで判定しています（祝日で木曜が週最終日の場合は問題ありません）。")
sectors = u.set_index("ticker")["sector"]
ind = T.indicators(p, sectors)  # 全銘柄のパラボリックSARなどを計算するため1〜2分かかる

## 3. 今週末の判定

In [ ]:
now = T.latest(ind, [TICKER], signal=SIGNAL, conditions=CONDS)
num = ["%R14", "RSI2", "RSI14", "25日線乖離%", "週間騰落%", "市場の週間騰落%", "業種平均との差%", "同業種で%R<-80の割合%"]
now[num] = now[num].round(1)
now.insert(0, "銘柄", NAME)
now.T

In [ ]:
r = now.loc[TICKER]
if not r["シグナル"]:
    print(f"■ {NAME}: シグナルなし（%R14 = {r['%R14']}）→ 見送り")
else:
    print(f"■ {NAME}: シグナルあり（%R14 = {r['%R14']}）  参考点 {int(r['参考点']):+d}")
print("\n同時に出ている条件:")
on = [c for c in CONDS if r[c.label]]
for c in on:
    print(f"   [{c.side}] {c.label}\n        分析時点の結果: {c.note}")
if not on:
    print("   なし")
print("\n※ シグナルが出ていない週の条件は参考表示です。")

## 4. 過去のシグナル時の成績（SOMPO、2008年〜）

### 4-1. 全体と、条件ごとの上昇確率
「あり」はその条件が同時に出ていたシグナル週、「なし」は出ていなかったシグナル週です。回数が少ない（目安として20回未満）ものは、確率が大きくぶれます。

In [ ]:
hist = T.history(p, ind, [TICKER], signal=SIGNAL, conditions=CONDS)
r_ = hist["翌週騰落率"]
print(f"シグナル {len(r_)} 回 / 翌週の上昇確率 {(r_ > 0).mean():.1%} / 平均 {r_.mean():+.2%} / 中央値 {r_.median():+.2%}（配当込み・コスト控除前）")
ct = T.conditional_table(hist, CONDS).drop(columns="ticker").set_index("条件")
ct.style.format({"あり_上昇確率": "{:.1%}", "なし_上昇確率": "{:.1%}"}, na_rep="—")

### 4-2. 参考点別の過去の成績
⚠️ 条件を選んだのと同じ期間での集計です。将来もこの通りになる保証はありません。

In [ ]:
sc = hist.groupby("参考点")["翌週騰落率"].agg(回数="size", 上昇確率=lambda x: (x > 0).mean(), 平均=lambda x: x.mean())
sc.style.format({"上昇確率": "{:.1%}", "平均": "{:+.2%}"})

### 4-3. 直近のシグナル週の一覧

In [ ]:
since = last - pd.DateOffset(years=RECENT_YEARS)
recent = hist[hist["判定日"] >= since].copy()
label = {c.key: f"[{c.side}]{c.label}" for c in CONDS}
view = recent[["判定日", "買い", "売り", SG.column, "翌週騰落率", "参考点"] + list(label)].rename(columns=label)
for c in ["判定日", "買い", "売り"]:
    view[c] = view[c].dt.date
rr = recent["翌週騰落率"]
print(f"直近{RECENT_YEARS}年: {len(rr)}回, 上昇確率 {(rr > 0).mean():.1%}, 平均 {rr.mean():+.2%}")
view.style.format({SG.column: "{:.1f}", "翌週騰落率": "{:+.2%}"}).map(
    lambda v: "background-color: #e6f4ea" if v is True else "", subset=list(label.values()))

## 5. 条件の検証（参考）

各条件がシグナル週の上昇確率を本当に変えていたかを、**SOMPO・他の保険株・全銘柄**で比べます。
- 差は「条件あり − 条件なし」の上昇確率（ポイント）です。前半は2008〜2017年、後半は2018年以降です。
- 3つの比較で向きがそろい、前半・後半とも同じ向きなら、比較的信頼できます。
- 全銘柄の z値は、同じ週の銘柄を独立に数えるため、市場全体の条件では大きく出すぎます。

In [ ]:
if CHECK_CONDITIONS:
    peers = [t for t in p.close.columns if sectors.get(t) == sectors.get(TICKER) and t != TICKER]
    eff = {
        NAME: T.condition_effect(p, ind, SIGNAL, CONDS, tickers=[TICKER]),
        f"他の{sectors.get(TICKER)}（{len(peers)}銘柄）": T.condition_effect(p, ind, SIGNAL, CONDS, tickers=peers),
        "全銘柄": T.condition_effect(p, ind, SIGNAL, CONDS),
    }
    tbl = pd.concat({k: v.set_index(["条件", "向き"])[["あり_回数", "差", "前半_差", "後半_差"]] for k, v in eff.items()}, axis=1)
    display(tbl.style.format("{:+.1f}", subset=[c for c in tbl.columns if "差" in c[1]], na_rep="—"))

### 5-1. 自分で条件を追加して検証する
`ind` には日次の指標（行＝日付、列＝銘柄）が入っています。条件を True/False の表として作り、`ind["flag_名前"]` に入れて `T.Condition("名前", ...)` を作ると、上と同じ方法で検証できます。例として「RSI(14) < 30」を試します。

In [ ]:
ind["flag_my_rsi14"] = ind["rsi14"] < 30
mine = [T.Condition("my_rsi14", "RSI(14) < 30（自作の例）", "?", "")]
if CHECK_CONDITIONS:
    display(pd.concat({k: T.condition_effect(p, ind, SIGNAL, mine, tickers=tk).set_index("条件")
                       for k, tk in [(NAME, [TICKER]), ("全銘柄", None)]}, axis=1).round(3))
    print("今週の状態:", bool(ind["flag_my_rsi14"].at[last, TICKER]))

## 6. チャート（直近1年）
▲ はシグナル週の判定日です（赤＝翌週上昇、灰＝翌週下落）。

In [ ]:
start = last - pd.DateOffset(years=1)
c = p.close[TICKER].loc[start:]
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax.plot(c.index, c, color="black", lw=1.2, label="終値")
for n, col in [(25, "tab:blue"), (75, "tab:orange"), (200, "tab:green")]:
    ax.plot(c.index, p.close[TICKER].rolling(n).mean().loc[start:], lw=0.9, color=col, label=f"{n}日線")
ev = hist[hist["判定日"] >= start]
ax.scatter(ev["判定日"], c.reindex(ev["判定日"]), marker="^", s=60,
           c=np.where(ev["翌週騰落率"] > 0, "tab:red", "tab:gray"), zorder=3, label="シグナル")
ax.set_title(f"{NAME}（{TICKER}）")
ax.legend(loc="upper left", fontsize=8)
ax.grid(alpha=0.3)
ax2.plot(c.index, ind["wr14"][TICKER].loc[start:], lw=0.8, color="tab:purple")
ax2.axhline(T.WR_THRESHOLD, color="red", lw=0.8, ls="--")
ax2.set_ylim(-100, 0)
ax2.set_ylabel("%R(14)")
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. 建玉と損切りの目安

**分析時点（全銘柄・SOMPO、2008〜2026/10）でのラインの比較**
- 期待値が最も高かったのは、**損切り・利確を置かずに最長1週間で決済**する形でした。
- 近いライン（損切り3〜5%、利確3〜6%）は成績を悪化させました。
- 損切りを置くなら **7%〜ATR×2** 程度の遠めに置くと、平均は1回あたり0.1〜0.2ポイント下がる一方、最悪の損失をおおむね-10%前後に抑えられました。

買値は、寄りで約定するまでは最新の終値を仮に使います。約定後は、1. の `ENTRY_PRICE` に買値を入れて再実行してください。窓を開けて損切りラインを飛び越えた場合は、ラインより不利な価格で約定します。

In [ ]:
entry = float(ENTRY_PRICE or p.close[TICKER].iloc[-1])
atr = T.trade_levels(p, TICKER, entry)["ATR14"]
lot = entry * 100
shares = int(CAPITAL // lot) * 100  # レバレッジ1倍
print(f"買値（{'入力値' if ENTRY_PRICE else '最新終値で仮置き'}）: {entry:,.1f} 円 / ATR(14): {atr:,.1f} 円（{atr / entry:.1%}）")
print(f"100株の約定代金: {lot:,.0f} 円")
if shares:
    print(f"保証金 {CAPITAL:,} 円・レバレッジ1倍での株数: {shares} 株（約定代金 {shares * entry:,.0f} 円）")
else:
    shares = 100
    print(f"⚠️ 保証金 {CAPITAL:,} 円ではレバレッジ1倍で100株を買えません。"
          f"100株ならレバレッジ約{lot / CAPITAL:.2f}倍（約定代金の{CAPITAL / lot:.0%}が保証金）になります。")
    print("   以下は100株で計算します。レバレッジをかけると、同じ値動きでも保証金に対する損益の割合が大きくなります。")
print(f"1週間（7日）持った場合の金利の目安: {shares * entry * INTEREST * 7 / 365:,.0f} 円（年率 {INTEREST:.1%} の仮定）")
rows = []
for name, stop in [("損切り7%", entry * 0.93), ("損切り ATR×2", entry - 2 * atr), ("損切り ATR×3", entry - 3 * atr)]:
    loss = (stop - entry) * shares
    rows.append({"案": name, "損切り価格": stop, "下落率": stop / entry - 1, "想定損失（円）": loss, "保証金に対して": loss / CAPITAL})
pd.DataFrame(rows).set_index("案").style.format(
    {"損切り価格": "{:,.1f}", "下落率": "{:+.1%}", "想定損失（円）": "{:+,.0f}", "保証金に対して": "{:+.1%}"})

## 8. 判断のメモ（例）

- シグナルがなければ見送る
- シグナルがあれば、参考点と 5. の検証結果を確認する（回数の少ない条件や、前半・後半で向きが違う条件は重視しない）
- 「－」の条件（株価が75日線より上＝上昇トレンド中の軽い押し目、市場全体が強い局面）が重なるときは慎重に
- 発注前に、翌週の決算発表・権利落ち・開示を各社IRやTDnetで確認する

毎週の判定と結果を記録しておくと、今後のデータで、このシグナルや条件が本当に効くかを確かめられます。